# Revenue Metrics
as of July 2026

In [0]:
%sql
--Revenue for Finance

select service_period_end
    ,SUM(invoice_amount)-SUM(refund_amount)
from databrick_by_databrick.default.invoices
where service_period_end = '2026-07-31'
group by 1;

In [0]:
%sql
--Revenue for Sales

select sum(annual_contract_value)
from databrick_by_databrick.default.opportunities
where stage = 'Closed Won' and date_trunc('MONTH', close_date) = '2026-07-01';

In [0]:
%sql
--Revenue for Product

select SUM(monthly_recurring_revenue)*12
from databrick_by_databrick.default.subscriptions
where subscription_status = 'Active';

In [0]:
%sql
--Revenue Governed
--HELP!!!

# Active Customers
as of July 2026

In [0]:
%sql
--Active Customers for Finance

select count(DISTINCT customer_id)
from databrick_by_databrick.default.invoices
where payment_date >= date_add('2026-07-31', -90) and payment_date <= '2026-07-31' and invoice_status = 'Paid';

In [0]:
%sql
--Active Customers for Sales

select count(DISTINCT customer_id)
from databrick_by_databrick.default.customers
where master_status IN ('Active', 'Trial');

In [0]:
%sql
--Active Customers for Product

select count(Distinct customer_id)
from databrick_by_databrick.default.product_usage
where usage_month = '2026-07-01' and sessions > 2;

In [0]:
%sql
--Active Customers Governed

with t as (
    select customer_id
        ,usage_month
        ,sessions
    from databrick_by_databrick.default.product_usage
    where usage_month = '2026-06-01' or usage_month = '2026-07-01'
)
select count(Distinct s.customer_id)
from databrick_by_databrick.default.subscriptions as s
join t on s.customer_id = t.customer_id
where subscription_status = 'Active' and sessions > 0;

# Conversion Rate
as of July 2026

In [0]:
%sql
--Conversion Rate for Marketing


In [0]:
%sql
--Conversion Rate for Sales

select SUM(case when stage = 'Closed Won' then 1 else 0 end) as won
    ,SUM(case when stage IN ('Closed Lost', 'Closed Won') then 1 else 0 end) as closed
    ,SUM(case when stage = 'Closed Won' then 1 else 0 end) / SUM(case when stage IN ('Closed Lost', 'Closed Won') then 1 else 0 end) as percent_won
from databrick_by_databrick.default.opportunities
where date_trunc('MONTH', close_date) = '2026-07-01';

In [0]:
%sql
--Conversion Rate for Executive

select COUNT(distinct converted_customer_id) as converted
    ,COUNT(distinct lead_id) as total_leads
    ,COUNT(distinct converted_customer_id) / COUNT(distinct lead_id) as percent_converted
from databrick_by_databrick.default.leads
where date_trunc('MONTH', lead_created_date) = '2026-07-01';

In [0]:
%sql
--Conversion Rate Governed

select SUM(case when stage = 'Closed Won' then 1 else 0 end) as won
    ,SUM(case when qualified_date IS NOT NULL then 1 else 0 end) as qualified
    ,SUM(case when stage = 'Closed Won' then 1 else 0 end)/SUM(case when qualified_date IS NOT NULL then 1 else 0 end) as percent_won
from databrick_by_databrick.default.leads as l
left join databrick_by_databrick.default.opportunities as o
on l.lead_id = o.lead_id
where date_trunc('MONTH', lead_created_date) = '2026-07-01';